# Lead Qualification Tool

**Build 1.1.2**

Scores a CSV of inbound leads against a frozen four-factor rubric, writes a
personalised outreach message for every qualified lead, and produces a report
a sales team can work from.

This notebook runs the pipeline end to end. Scoring, messaging and reporting
live in the three `.py` modules; the notebook adds the token pacer and the
tiering call, and wires everything together.

| Block | Runs | Produces |
|---|---|---|
| 1. Setup | once per session | config, API key, module check |
| 2. Pipeline | once per session | `run_pipeline()` and the token pacer |
| 3. **RUN** | once per file | the report, six files on disk |
| 4. Report | after a run | the sales-team view |
| 5. Download | after a run | every file written |

**Required in the session folder:** `config.yaml`, `rubric_scorer.py`,
`message_generator.py`, `report_builder.py`, and your leads CSV.
A Groq API key must be set in Colab secrets under `GROQ_API_KEY`.

## 1. Setup

Loads the frozen config, resolves the API key, and confirms every module is
present. Prints what it loaded so the run is self-describing — a report is
only readable if you can tell which rubric version and which batch sizes
produced it.

In [26]:
!pip -q install pyyaml pandas

import json, os, re, time
from pathlib import Path

import pandas as pd
import requests

import rubric_scorer as R          # stage 1 — scoring, guardrails, ranking
import message_generator as MG     # stage 2 — outreach messages
import report_builder as RB        # stage 3 — the report

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

WORKDIR = Path(".")

# --- modules and config must be present before anything else ---------------
REQUIRED = ["config.yaml", "rubric_scorer.py", "message_generator.py", "report_builder.py"]
absent = [f for f in REQUIRED if not (WORKDIR / f).exists()]
if absent:
    raise FileNotFoundError(
        f"missing from the session folder: {absent}. Upload them with the folder "
        f"icon on the left, then re-run this cell."
    )

CFG = R.load_config(WORKDIR / "config.yaml")
LLM, MSG = CFG["llm"], CFG["llm_messages"]


# --- API key: resolved once, here, so a missing key fails before any work ---
def resolve_api_key() -> str | None:
    """Environment first, then Colab secrets. Named in config, not hardcoded."""
    key = os.environ.get(LLM["api_key_env"])
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get(LLM["api_key_env"])
    except Exception:
        return None


API_KEY = resolve_api_key()

print(f"rubric        {CFG['meta']['version']}   (frozen: {CFG['meta']['calibration_basis']})")
print(f"build         {RB.BUILD}")
print(f"model         {LLM['model']}")
print()
print(f"cutoff        {CFG['decision']['qualify_cutoff']}  "
      f"review margin +/-{CFG['decision']['review_margin']}")
print(f"weights       " + "  ".join(
    f"{k}={v['score_weight']}" for k, v in CFG["factors"].items()))
print(f"blend         fit {CFG['priority']['blend']['fit']} / "
      f"urgency {CFG['priority']['blend']['urgency']}")
print()
# Two batch_size keys exist and they count different things. Printing both,
# labelled, is the only way a rate in the report is readable later.
print(f"llm.batch_size           {LLM['batch_size']:>3}   unique INDUSTRY STRINGS per tiering call  "
      f"(temp {LLM['temperature']})")
print(f"llm_messages.batch_size  {MSG['batch_size']:>3}   LEADS per message call                    "
      f"(temp {MSG['temperature']}, max_tokens {MSG['max_tokens']})")
print()
print(f"API key ({LLM['api_key_env']}) present: {bool(API_KEY)}")
if not API_KEY:
    print(f"  -> set it in Colab secrets under {LLM['api_key_env']!r} before running block 3")

rubric        1.1-frozen-on-training   (frozen: leads_training.csv (n=29))
build         1.1.2
model         openai/gpt-oss-120b

cutoff        7.0  review margin +/-0.25
weights       source=0.75  industry=1.75  company_size=1.25  recency=1.0
blend         fit 0.8 / urgency 0.2

llm.batch_size            25   unique INDUSTRY STRINGS per tiering call  (temp 0.0)
llm_messages.batch_size    4   LEADS per message call                    (temp 0.7, max_tokens 4500)

API key (GROQ_API_KEY) present: True


## 2. Pipeline

All logic lives in the three `.py` modules. This block wires them together and
adds the two things that belong to the *account* rather than to any module:
the free-tier token pacer, and the industry tiering call.

**Not purely deterministic.** The scoring *arithmetic* is deterministic and
reproducible. The industry tier is a cached model judgment — classified once
per unique industry string, then read from `industry_tier_cache.json` forever
after. Message text is generated at temperature 0.7 and is not reproducible
across files, only across re-runs of the same file.

**Why the pacer is here and not in a module.** The 8,000 TPM ceiling is a
property of the Groq free tier and this account, not of the message stage.
`generate_messages(client=...)` is injectable precisely so the pacer can wrap
the call without editing `message_generator.py`. Both LLM boundaries — tiering
and messaging — charge the same ledger, because the provider counts them
against the same window.

In [27]:
# ---------------------------------------------------------------------------
# Token pacer — one ledger, both LLM boundaries
#
# Groq's on_demand TPM ceiling for this model is 8,000. Two earlier runs
# failed against it, and neither failure was a pipeline defect: a call that
# returns finish_reason=length with an empty body still burns the full
# completion cap. The pacer meters a 60s sliding window of ACTUAL reported
# usage and refuses to start a call that would not fit.
# ---------------------------------------------------------------------------
TPM_LIMIT = 8000            # provider's stated ceiling for this model + tier
TPM_HEADROOM = 0.85         # room for estimate error between report and reality
WINDOW_S = 60.0


class TokenPacer:
    """Sliding-window token budget. Shared by tiering and messaging because
    the provider charges both against the same per-minute window."""

    def __init__(self, limit=TPM_LIMIT, headroom=TPM_HEADROOM, window=WINDOW_S):
        self.budget = limit * headroom
        self.window = window
        self._spend = []        # (finished_at, total_tokens)

    def gate(self, ceiling: int, log=print) -> None:
        """Block until `ceiling` more tokens fit in the window."""
        while True:
            now = time.time()
            self._spend[:] = [(t, n) for t, n in self._spend if now - t < self.window]
            used = sum(n for _, n in self._spend)
            # An empty window always proceeds: a single call cannot be paced
            # below its own cost. This is also the loop's exit condition.
            if not self._spend or used + ceiling <= self.budget:
                return
            wait = self.window - (now - self._spend[0][0]) + 1
            log(f"[rate] window {used:.0f} + next call ~{ceiling} exceeds "
                f"{self.budget:.0f} - sleeping {wait:.0f}s")
            time.sleep(wait)

    def charge(self, usage: dict | None, ceiling: int) -> int:
        """Record actual usage; fall back to the ceiling when none is reported."""
        usage = usage or {}
        total = (usage.get("total_tokens")
                 or usage.get("prompt_tokens", 0) + usage.get("completion_tokens", 0)
                 or ceiling)
        self._spend.append((time.time(), total))
        return total

    def used(self) -> float:
        now = time.time()
        return sum(n for t, n in self._spend if now - t < self.window)


PACER = TokenPacer()


# ---------------------------------------------------------------------------
# Stage 1 LLM boundary — industry tier normalisation
#
# Runs once over the set of UNIQUE industry strings, never once per lead, and
# caches to disk. The same string must receive the same tier in every file, so
# the cache is shared across files by design — unlike the message cache, which
# is deliberately per-file.
#
# Temperature 0 and fail-closed on truncation: a partially-classified batch is
# discarded rather than half-trusted. A string the model declines to tier is
# treated as missing, never defaulted to a middle tier.
#
# [NOTE] TIER_PROMPT is business text living outside config. This is the one
# documented place the "business text lives in config" rule does not hold;
# it is carried verbatim from the training run so the tiers stay comparable.
# ---------------------------------------------------------------------------
TIER_PROMPT = """You classify industry labels for a B2B SaaS vendor.

The vendor sells a CRM/ERP suite for customer onboarding, engagement and
post-sale support. It fits a prospect well when that prospect's own customer
operations are digital and generate structured customer records. It fits
poorly when the prospect's operations are dominated by physical assets and
movement.

Assign each label exactly one tier:
  tier_1 - digital customer interactions are core to how the business runs
  tier_2 - mixed: meaningful digital customer operations alongside physical
           or offline delivery
  tier_3 - physical assets, movement or field operations dominate; few
           structured digital customer records

Labels:
{labels}

Return ONLY a JSON array, no prose and no markdown fences. One object per
label, same count and same spelling as the input:
[{{"industry": "<label exactly as given>", "tier": "tier_1|tier_2|tier_3"}}]"""


class TierLLMError(Exception):
    """Carries whether the failure is worth retrying. Auth, permission and
    bad-request errors are not — retrying them just burns the budget."""

    def __init__(self, message, retryable):
        self.retryable = retryable
        super().__init__(message)


def call_llm_tiering(prompt: str, api_key: str) -> dict:
    """-> {content, usage}. Paced, and the provider's own error body is
    surfaced rather than swallowed into a bare status code."""
    ceiling = LLM["max_tokens"] + 1000
    PACER.gate(ceiling)
    try:
        resp = requests.post(
            LLM["base_url"],
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {api_key}",
                # Default library user agents get fingerprinted and blocked by
                # Cloudflare at the provider edge (error 1010) before the
                # request ever reaches the API.
                "User-Agent": "lead-intelligence/1.1.2",
            },
            json={
                "model": LLM["model"],
                "temperature": LLM["temperature"],
                "max_tokens": LLM["max_tokens"],
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=LLM["request_timeout_seconds"],
        )
    except requests.RequestException as e:
        raise TierLLMError(f"transport: {e}", retryable=True) from None

    if resp.status_code != 200:
        if resp.status_code == 429:
            PACER.charge(None, ceiling)
        # The server's own explanation lives in the body. Surfacing it is the
        # difference between "HTTP 403" and "model_not_found".
        raise TierLLMError(f"HTTP {resp.status_code}: {resp.text[:400]}",
                           retryable=resp.status_code == 429 or resp.status_code >= 500)

    body = resp.json()
    PACER.charge(body.get("usage"), ceiling)
    choice = body["choices"][0]
    if choice.get("finish_reason") == "length":
        # Fail closed: a truncated classification array cannot be trusted.
        raise TierLLMError("response truncated: finish_reason=length", retryable=False)
    return {"content": choice["message"]["content"], "usage": body.get("usage") or {}}


def extract_tier_array(text: str):
    """Returns a list, or a parse_error dict. Never raises."""
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    start, end = cleaned.find("["), cleaned.rfind("]")
    if start == -1 or end == -1:
        return {"parse_error": "no JSON array found", "raw": text[:300]}
    try:
        return json.loads(cleaned[start:end + 1])
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw": cleaned[start:end + 1][:300]}


def classify_batch(labels, api_key, log=print):
    """-> (mapping, diagnostics). Retries transport and parse failures only,
    bounded by llm.max_retries. Alignment is by label, never by position."""
    prompt = TIER_PROMPT.format(labels="\n".join(f"- {l}" for l in labels))
    diag = {"n_in": len(labels), "attempts": 0, "errors": [], "n_out": 0,
            "n_aligned": 0, "misaligned": list(labels),
            "prompt_tokens": 0, "completion_tokens": 0}

    for attempt in range(1, LLM["max_retries"] + 1):
        diag["attempts"] = attempt
        try:
            result = call_llm_tiering(prompt, api_key)
        except TierLLMError as e:
            diag["errors"].append(str(e))
            if not e.retryable:
                break
            time.sleep(LLM["retry_backoff_seconds"] * attempt)
            continue

        u = result["usage"]
        diag["prompt_tokens"] += int(u.get("prompt_tokens", 0))
        diag["completion_tokens"] += int(u.get("completion_tokens", 0))

        parsed = extract_tier_array(result["content"])
        if isinstance(parsed, dict):
            diag["errors"].append(f"parse: {parsed['parse_error']}")
            time.sleep(LLM["retry_backoff_seconds"] * attempt)
            continue

        mapping = {
            str(e.get("industry", "")).strip(): str(e.get("tier", "")).strip()
            for e in parsed
            if isinstance(e, dict)
            and str(e.get("tier", "")).strip() in set(CFG["factors"]["industry"]["tier_scores"])
        }
        recovered = {l: mapping[l] for l in labels if l in mapping}
        diag.update(n_out=len(parsed), n_aligned=len(recovered),
                    misaligned=sorted(set(labels) - set(recovered)))
        if recovered:
            return recovered, diag
        diag["errors"].append("zero labels aligned")
        time.sleep(LLM["retry_backoff_seconds"] * attempt)

    return {}, diag


def tier_industries(df, api_key, log=print) -> tuple[dict, dict]:
    """Cache-first. A live call is made only for strings never seen before."""
    cache_path = WORKDIR / CFG["factors"]["industry"]["normalisation"]["cache_path"]
    tier_map = json.loads(cache_path.read_text()) if cache_path.exists() else {}

    unique = sorted({s for s in df["industry"].dropna().astype(str).str.strip() if s})
    todo = [s for s in unique if s not in tier_map]
    log(f"  industries: n={len(unique)} unique | cached {len(unique) - len(todo)} | "
        f"to classify {len(todo)}")

    meta = {"n_strings_total": len(unique), "n_from_cache": len(unique) - len(todo),
            "n_classified_live": 0, "batches": 0, "parse_failures": 0,
            "unclassified": [], "batch_size_key": "llm.batch_size",
            "batch_size": LLM["batch_size"],
            "tokens": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}}

    if todo:
        if not api_key:
            raise RuntimeError(
                f"{len(todo)} industry strings need classifying and no API key is set. "
                f"Add {LLM['api_key_env']!r} to Colab secrets and re-run block 1."
            )
        bs = int(LLM["batch_size"])
        for i in range(0, len(todo), bs):
            mapping, diag = classify_batch(todo[i:i + bs], api_key, log)
            meta["batches"] += 1
            meta["parse_failures"] += sum(1 for e in diag["errors"] if e.startswith("parse"))
            meta["tokens"]["prompt_tokens"] += diag["prompt_tokens"]
            meta["tokens"]["completion_tokens"] += diag["completion_tokens"]
            tier_map.update(mapping)
            log(f"  tiering batch {meta['batches']}: sent={diag['n_in']} "
                f"aligned={diag['n_aligned']} attempts={diag['attempts']}"
                + (f" | {diag['errors']}" if diag["errors"] else ""))
        meta["n_classified_live"] = len([s for s in todo if s in tier_map])
        meta["unclassified"] = [s for s in todo if s not in tier_map]
        cache_path.write_text(json.dumps(tier_map, indent=2, sort_keys=True))
        log(f"  tier cache written: {cache_path} (n={len(tier_map)})")

    meta["tokens"]["total_tokens"] = (meta["tokens"]["prompt_tokens"]
                                      + meta["tokens"]["completion_tokens"])
    # A string the model would not tier scores as missing, never as a middle
    # tier — that would assign a real score to something never classified.
    if meta["unclassified"]:
        log(f"  [warn] unclassified, will score as missing: {meta['unclassified']}")
    return tier_map, meta


# ---------------------------------------------------------------------------
# Stage 2 LLM boundary — the paced message client
#
# Wraps MG.call_llm_messages without editing the module. On a 429 it charges
# the ceiling, waits out the window, then re-raises so the module's own
# transport counter still bounds the retry loop at 3 attempts.
# ---------------------------------------------------------------------------
def paced_message_client(prompt, cfg, api_key):
    ceiling = int(cfg["llm_messages"]["max_tokens"]) + 1000
    PACER.gate(ceiling)
    try:
        result = MG.call_llm_messages(prompt, cfg, api_key)
    except MG.MessageLLMError as e:
        if "429" in str(e):
            PACER.charge(None, ceiling)
            print(f"[rate] 429 despite pacing - sleeping {WINDOW_S:.0f}s before the "
                  f"transport retry")
            time.sleep(WINDOW_S)
        raise
    total = PACER.charge(result.get("usage"), ceiling)
    print(f"[rate] call used {total} tokens | window {PACER.used():.0f}/{PACER.budget:.0f}")
    return result


# ---------------------------------------------------------------------------
# The pipeline
# ---------------------------------------------------------------------------
def validate_input(path: Path) -> pd.DataFrame:
    """Hard stop naming the problem, not a KeyError three frames deep."""
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is not in the session folder. Upload it with the folder icon "
            f"on the left, then re-run this cell."
        )
    df = pd.read_csv(path)
    required = list(CFG["missing_data"]["completeness_fields"])
    absent = [c for c in required if c not in df.columns]
    if absent:
        raise ValueError(
            f"{path.name} is missing required column(s): {absent}. "
            f"Required: {required}. Found: {list(df.columns)}"
        )
    if len(df) == 0:
        raise ValueError(f"{path.name} has headers but no rows.")
    return df


def run_pipeline(input_csv: str, api_key: str | None = None, log=print):
    """CSV in, report out. Every step has an exit condition.

    validate -> tier -> score/guardrails/rank -> filter QUALIFIED
             -> generate messages -> build/write report -> list files
    """
    api_key = api_key if api_key is not None else API_KEY
    path = WORKDIR / input_csv
    stem = RB.derive_stem(path, CFG)
    paths = RB.output_paths(CFG, stem, WORKDIR)
    t0 = time.time()

    log(f"── {path.name} ─────────────────────────────────────────")
    df = validate_input(path)
    log(f"  validated: n={len(df)} rows, {len(df.columns)} columns")

    # --- stage 1 -----------------------------------------------------------
    tier_map, s1 = tier_industries(df, api_key, log)
    leads, processing_date = R.score_dataframe(df, CFG, tier_map)
    s1["processing_date"] = processing_date
    counts = {}
    for l in leads:
        counts[l.decision] = counts.get(l.decision, 0) + 1
    log(f"  scored: n={len(leads)} | processing_date {processing_date} | {counts}")

    R.to_frame(leads).to_csv(paths["scored_table"], index=False)
    paths["stage1_report"].write_text(json.dumps(
        {"stage": "1 - scoring", "source_file": path.name,
         "rubric_version": CFG["meta"]["version"], "build": RB.BUILD,
         "processing_date": str(processing_date), "n_leads": len(leads),
         "decisions": counts, "tiering": s1}, indent=2, default=str))

    # --- stage 2 -----------------------------------------------------------
    qualified = [l for l in leads if l.decision == "QUALIFIED"]
    msg_report = None
    if not qualified:
        # Zero-qualified is a valid outcome, not an error. Nothing to send.
        log("  no qualified leads - message generation skipped")
    elif not api_key:
        raise RuntimeError(
            f"{len(qualified)} qualified leads need messages and no API key is set. "
            f"Add {LLM['api_key_env']!r} to Colab secrets and re-run block 1."
        )
    else:
        log(f"  generating messages for n={len(qualified)} qualified leads "
            f"(batch_size {MSG['batch_size']}, temp {MSG['temperature']})")
        msg_report = MG.generate_messages(
            qualified, CFG, api_key=api_key, source_file=path.name,
            cache_path=paths["message_cache"], client=paced_message_client, log=log,
        )
        paths["message_report"].write_text(json.dumps(msg_report, indent=2, default=str))
        log(f"  messages: generated {msg_report['n_generated']} | "
            f"cached {msg_report['n_cached']} | failed {msg_report['n_failed']} "
            f"of n={msg_report['n_leads']}")

    # --- stage 3 -----------------------------------------------------------
    report = RB.build_report(leads, CFG, source_file=path.name,
                             stage1_meta=s1, stage2_report=msg_report, log=log)
    written = RB.write_report(report, CFG, stem, WORKDIR)

    all_written = [str(paths["scored_table"]), str(paths["stage1_report"])]
    if msg_report is not None:
        all_written += [str(paths["message_report"]), str(paths["message_cache"])]
    all_written += written
    report["files_written"] = all_written

    log("")
    log(f"  done in {time.time() - t0:.1f}s. files written:")
    for f in all_written:
        log(f"    {f}")
    return report, leads


print("pipeline ready:  report, leads = run_pipeline(INPUT_CSV)")

pipeline ready:  report, leads = run_pipeline(INPUT_CSV)


## 3. RUN

Set `INPUT_CSV` and run this cell. It does everything: scores every lead,
writes outreach messages for the qualified ones, builds the report and writes
every file. Nothing else needs running.

Output names all derive from the input file's stem, so no run can overwrite
another run's record — and a new CSV automatically starts with a fresh message
cache, with no clearing step and no flag. Re-running the *same* CSV hits its
own cache and reproduces the same messages.

In [28]:
# ── RUN ────────────────────────────────────────────────────────────────────
# 1. Upload your leads CSV to the session (folder icon, left).
#      Required columns: name, company, company_size, industry,
#                        source, last_interaction_date
# 2. Set INPUT_CSV below.
# 3. Run this cell.
#
# Requires a Groq API key in Colab secrets under GROQ_API_KEY.
# Industry strings not already in industry_tier_cache.json are classified
# live on first contact, so the first run on a new file makes more calls
# than later runs on the same file.

INPUT_CSV = "leads_sample_50.csv"

report, leads = run_pipeline(INPUT_CSV)

── leads_sample_50.csv ─────────────────────────────────────────
  validated: n=50 rows, 6 columns
  industries: n=26 unique | cached 0 | to classify 26
  tiering batch 1: sent=25 aligned=25 attempts=1
  tiering batch 2: sent=1 aligned=1 attempts=1
  tier cache written: industry_tier_cache.json (n=75)
  scored: n=50 | processing_date 2024-01-20 | {'QUALIFIED': 23, 'REJECTED': 23, 'REVIEW': 4}
  generating messages for n=23 qualified leads (batch_size 4, temp 0.7)
[msg] leads n=23 | cached n=0 | to generate n=23
[rate] window 1786 + next call ~5500 exceeds 6800 - sleeping 60s
[rate] call used 3533 tokens | window 3533/6800
[msg] v1_value  batch 1/2  attempt 1   sent=4   recovered=4   missing=0
        finish=stop  truncated_tail=no  tokens p746/c2787  6.2s
[msg] v1_value  batch 1/2  OK  4/4
[rate] window 3533 + next call ~5500 exceeds 6800 - sleeping 61s
[rate] call used 2172 tokens | window 2172/6800
[msg] v1_value  batch 2/2  attempt 1   sent=1   recovered=1   missing=0
        finish

## 4. Report

The sales-team view. QUALIFIED first, then the leads flagged for a human, then
what was not pursued and why.

`priority_rank` is a **queue position, not a qualification ranking**. The
decision comes from `fit_score` and the queue position from `priority_score`,
and the two interleave — a REVIEW lead can outrank a REJECTED one. Grouping by
decision is what makes that read correctly.

### Report Summary

In [29]:
## Run this cell for report summary

RB.render_report(report, max_rows=15)

══════════════════════════════════════════════════════════════════
  LEAD INTELLIGENCE REPORT - leads_sample_50.csv
  Rubric 1.1-frozen-on-training - build 1.1.2
  Scored against 2024-01-20
══════════════════════════════════════════════════════════════════

  PROCESSED   50 leads    QUALIFIED     23  (46.0%)
  SCORED      50 leads    REVIEW         4  (8.0%)
                          REJECTED      23  (46.0%)

  At 1,200 inbound/month this projects to ~552 qualified
  and ~96 to human review, against a team currently working
  ~60. Projection from one file, n=50.

──── PRIORITY QUEUE - QUALIFIED ──────────────────────────────────
  rank  lead  company                     size  industry       score  band
     1  L022  Wolverton Systems          4,297  Subscription M  9.0  HIGH
     2  L042  Pinehurst Exchange         2,152  Streaming Plat  8.7  HIGH
     3  L038  Westbay Bureau               329  Subscription M  8.2  HIGH
     4  L003  Eastgate Collective          360  Identity Manag  8

### Report Table

In [30]:
# Full queue as a table. The CSV on disk carries every column; this is the
# working subset a rep actually scans.
rows = [
    {"decision": r["decision"], "rank": r["priority_rank"], "lead_id": r["lead_id"],
     "company": r["company"], "size": r["company_size"], "industry": r["industry"],
     "source": r["source"], "score": r["qualification_score"],
     "band": r["priority_band"],
     "reasons": ";".join(r["outcome_reason_codes"]),
     "variant": r["message_variant"]}
    for g in report["queue"] for r in g["leads"]
]
queue_df = pd.DataFrame(rows)
print(f"n={len(queue_df)} leads | "
      f"{report['summary']['decisions']} | "
      f"qualified {report['summary']['qualified_pct']['value']}% of "
      f"n={report['summary']['qualified_pct']['n']} "
      f"({report['summary']['qualified_pct']['denominator']})")
queue_df

n=50 leads | {'QUALIFIED': 23, 'REVIEW': 4, 'REJECTED': 23} | qualified 46.0% of n=50 (all leads processed)


,decision,rank,lead_id,company,size,industry,source,score,band,reasons,variant
0,QUALIFIED,1,L022,Wolverton Systems,4297.0,Subscription Media,Content download,9.0,HIGH,,v1_value
1,QUALIFIED,2,L042,Pinehurst Exchange,2152.0,Streaming Platform,Inbound demo request,8.7,HIGH,,v1_value
2,QUALIFIED,3,L038,Westbay Bureau,329.0,Subscription Media,Content download,8.2,HIGH,,v2_engagement
3,QUALIFIED,4,L003,Eastgate Collective,360.0,Identity Management,Inbound demo request,8.2,HIGH,,v2_engagement
4,QUALIFIED,5,L044,Quarrymount Exchange,393.0,Gaming,Content download,8.2,HIGH,,v2_engagement
5,QUALIFIED,6,L023,Amberline Holdings,389.0,Digital Banking,Inbound demo request,8.2,HIGH,,v2_engagement
6,QUALIFIED,7,L009,Orchard Holdings,380.0,Payments,Inbound demo request,8.2,HIGH,,v2_engagement
7,QUALIFIED,8,L013,Quill Foundry,413.0,Gaming,Webinar attendee,8.2,HIGH,,v2_engagement
8,QUALIFIED,9,L040,Alderway Group,436.0,Digital Banking,Webinar attendee,8.2,HIGH,,v2_engagement
9,QUALIFIED,10,L046,Northwind Bureau,344.0,Developer Tools,Inbound demo request,8.1,HIGH,,v2_engagement


## 5. Download

Every file this run wrote. The two `_output_report` files are the
deliverable; the rest are the intermediate records each stage persisted.

In [ ]:
print("files written by this run:")
for f in report["files_written"]:
    p = Path(f)
    print(f"  {p.name:<42} {p.stat().st_size:>9,} bytes" if p.exists()
          else f"  {p.name:<42} MISSING")

try:
    from google.colab import files
    for f in report["files_written"]:
        files.download(f)
except Exception as e:
    print(f"\nnot in Colab or download blocked ({type(e).__name__}) - "
          f"download manually from the folder pane on the left.")

files written by this run:
  full_scale_100_scored.csv                     51,373 bytes
  full_scale_100_run_report.json                   639 bytes
  full_scale_100_message_run_report.json        29,388 bytes
  full_scale_100_message_cache.json             25,181 bytes
  full_scale_100_output_report.json            403,057 bytes
  full_scale_100_output_report.csv              30,870 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>